<a href="https://colab.research.google.com/github/A-Kuo/Language-Model-Hallucination-Detection-via-Entropy-Divergence/blob/main/colab/quick_cpu_validation.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# AED Quick CPU Validation
## Pipeline Test — No GPU Required

**Purpose:** Validate the entire pipeline works without GPU (for CI/testing)  
**Runtime:** ~3 minutes on CPU  
**Model:** GPT-2 (124M params) — smaller than Pythia-160m  
**Dataset:** 50 synthetic factual pairs (balanced)  

**⚠️ Note:** Results will be worse (AUROC ~0.5-0.7) than GPU benchmark because:
- GPT-2 is smaller (124M vs 160M params)
- Synthetic data is limited (50 vs 500 samples)
- This is for **pipeline validation only**, not paper results

**For paper results:** Use [GPU Benchmark](gpu_benchmark.ipynb) on T4 GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    print('✓ Running on CPU (expected for this notebook)')
else:
    print('⚠️  GPU detected — you could run the full GPU benchmark instead')

In [ ]:
%pip install -q transformers numpy
print('✓ Dependencies installed')

In [ ]:
import os, sys
REPO = 'Language-Model-Hallucination-Detection-via-Entropy-Divergence'
if not os.path.exists(REPO):
    !git clone https://github.com/A-Kuo/{REPO}.git
else:
    !git -C {REPO} pull
os.chdir(REPO)
sys.path.insert(0, 'v1')
print('✓ Repo ready:', os.getcwd())

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ── Run quick CPU benchmark ───────────────────────────────────────────────
# Model: gpt2 (124M params, 12 layers, 12 heads)
# Mode: quick (synthetic 50-sample dataset)
# Expected: AUROC ~0.5-0.7 (pipeline validation only)
# ──────────────────────────────────────────────────────────────────────────
!python v1/run_experiment.py \
    --mode quick \
    --model gpt2 \
    --output results/benchmark_results_cpu.json

In [ ]:
import json

with open('results/benchmark_results_cpu.json') as f:
    r = json.load(f)

print('='*60)
print('  CPU QUICK MODE RESULTS — Pipeline Validation')
print('='*60)
print(f'  Model:          {r["model_name"]}')
print(f'  Device:         {r["device"]}')
print(f'  Samples:        {r["num_samples"]} (synthetic)')
print()
print(f'  AED BiLSTM AUROC:      {r["aed_auroc"]:.4f}')
print(f'  LogReg Baseline AUROC: {r["logreg_auroc"]:.4f}')
print(f'  Latency:               {r["aed_latency_ms"]:.1f} ms/sample')
print('='*60)
print()
print('⚠️  NOTE: These are NOT paper-quality results.')
print('   GPT-2 is smaller than Pythia-160m.')
print('   Synthetic data (50 samples) vs HaluEval (500).')
print('   For paper: run gpu_benchmark.ipynb on T4 GPU')
print()
print('✓ Pipeline execution: SUCCESS')
print('  If AUROC is 0.5-0.7 and no errors → pipeline is working')

In [ ]:
# ── Diagnostic plots ────────────────────────────────────────────────────
# Show per-layer entropy patterns even on synthetic data

import numpy as np
import matplotlib.pyplot as plt
from run_experiment import extract_features, QUICK_PAIRS
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print('Loading GPT-2 for visualization...')
tokenizer = AutoTokenizer.from_pretrained('gpt2')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained('gpt2', output_attentions=True)
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

num_layers = model.config.n_layer
correct_entropy = np.zeros(num_layers)
halluc_entropy = np.zeros(num_layers)
correct_count = halluc_count = 0

for ctx, cont, label in QUICK_PAIRS:
    feats = extract_features(model, tokenizer, ctx, cont, device)
    if label == 0:
        correct_entropy += feats[:num_layers]
        correct_count += 1
    else:
        halluc_entropy += feats[:num_layers]
        halluc_count += 1

fig, ax = plt.subplots(figsize=(10, 6))
layers = range(1, num_layers + 1)
ax.plot(layers, correct_entropy/correct_count, 'b-o', label='Correct')
ax.plot(layers, halluc_entropy/halluc_count, 'r-o', label='Hallucinated')
ax.set_xlabel('Layer')
ax.set_ylabel('Mean Attention Entropy')
ax.set_title('Per-Layer Entropy (GPT-2, Synthetic Data)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig('results/figure_cpu_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: results/figure_cpu_validation.png')

In [ ]:
# ── Download results ───────────────────────────────────────────────────────
from google.colab import files
import os

for fname in ['results/benchmark_results_cpu.json', 'results/figure_cpu_validation.png']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'✓ Downloaded: {fname}')
    else:
        print(f'  Not found: {fname}')

print()
print('Next steps:')
print('  1. If validation passed → Run gpu_benchmark.ipynb for paper results')
print('  2. If errors occurred → Check error messages above')